Capstone Project: Deep Learning for NLP

CONTRIBUTION - INDIVIDUAL

GITHUB -

In this capstone project, students will embark on an exciting journey to create an Industry-Specific Large Language Model (LLM) Bot using state-of-the-art pre-trained models from sources like Hugging Face. The primary objective is to build an intelligent bot that can effectively engage with users by answering questions and providing insights specific to a chosen industry. This project will not only enhance your technical skills but also provide a deep understanding of the chosen industry's nuances, challenges, and trends.

In this project, we developed an industry-specific chatbot for the retail banking sector using a fine-tuned large language model (LLM). The primary goal was to create an AI assistant capable of answering real customer queries about banking services such as account balance checks, card activation, ATM location, loan applications, and more.

We used the publicly available Bitext Retail Banking Chatbot Dataset, which includes thousands of real-world banking prompts and professional, human-like responses. The model was fine-tuned using Google’s FLAN-T5-small on cleaned Q&A pairs, and deployed using a Gradio web interface for user-friendly interaction.

This chatbot demonstrates how LLMs can be effectively adapted to specialized industries like banking, delivering fast, informative, and polite responses — similar to a live customer support agent.



In [1]:
pip install transformers datasets scikit-learn gradio


In [3]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("/content/bitext-retail-banking-llm-chatbot-training-dataset.csv")

# Keep only needed columns and drop missing values
df_clean = df[['instruction', 'response']].dropna()
df_clean = df_clean[df_clean['instruction'].str.strip() != ""]
df_clean = df_clean[df_clean['response'].str.strip() != ""]
df_clean = df_clean.drop_duplicates(subset=['instruction', 'response'])

# Rename for model input/output
df_clean = df_clean.rename(columns={"instruction": "input", "response": "output"})

# Preview
df_clean.sample(3)


,input,output
23281,"I want to transfer money, Iwould like to make ...",I'm here to assist you with making a money tra...
4688,"what to do if I have olst my Amex, help me to ...",I'm really sorry to hear that you've lost your...
16527,"I don't have an online account , will you help...",I would be more than happy to assist you in op...


In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_clean, test_size=0.1, random_state=42)


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
from datasets import Dataset

# Convert to Hugging Face format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenization function
def tokenize(example):
    input_enc = tokenizer(example["input"], padding="max_length", truncation=True, max_length=128)
    target_enc = tokenizer(example["output"], padding="max_length", truncation=True, max_length=128)
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

# Tokenize datasets
train_tokenized = train_dataset.map(tokenize)
test_tokenized = test_dataset.map(tokenize)


Map:   0%|          | 0/22990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2555 [00:00<?, ? examples/s]

In [14]:
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./banking_bot",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    num_train_epochs=1,
    eval_strategy="epoch"
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer,
    data_collator=collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.845883,0.729706


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5748, training_loss=0.8525752609773236, metrics={'train_runtime': 1165.7986, 'train_samples_per_second': 19.72, 'train_steps_per_second': 4.931, 'total_flos': 1068405675786240.0, 'train_loss': 0.8525752609773236, 'epoch': 1.0})

In [15]:
# prompt: give testing accuracy training accuracy and validation accuracy

# After trainer.train()
import json

# Function to extract metrics from logs
def get_metrics(log_history):
    train_loss = None
    eval_loss = None
    # Find the final training loss
    for log in log_history:
        if 'loss' in log:
            train_loss = log['loss']
        if 'eval_loss' in log:
            eval_loss = log['eval_loss']
    return train_loss, eval_loss

train_loss, eval_loss = get_metrics(trainer.state.log_history)

print(f"Training Loss (last logged): {train_loss}")
print(f"Validation Loss (last epoch): {eval_loss}")

# To get evaluation metrics on the test set after training
eval_results = trainer.evaluate(test_tokenized)
print(f"Test Evaluation Results: {eval_results}")

# Note: For accuracy in seq2seq models, it's usually not a single number
# like classification accuracy. Metrics like ROUGE, BLEU are common.
# If you need specific accuracy-like metrics, you would need to implement
# or use a library for sequence generation evaluation metrics and potentially
# override the compute_metrics function in the Trainer or perform manual evaluation.
# The default evaluation in Seq2SeqTrainer provides loss.



Training Loss (last logged): 0.845883056640625
Validation Loss (last epoch): 0.7297060489654541


Training Loss,Validation Loss,Epoch
0.845883,0.729706,1


Test Evaluation Results: {'eval_loss': 0.7297060489654541}


In [16]:
model.save_pretrained("banking-chatbot")
tokenizer.save_pretrained("banking-chatbot")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('banking-chatbot/tokenizer_config.json', 'banking-chatbot/tokenizer.json')

In [17]:
import torch

def chat(prompt):
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Tokenize and move tensors to the same device as model
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate response
    with torch.no_grad():
        output = model.generate(**inputs, max_length=100)

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [18]:
chat("How can I activate my debit card?")


'I\'m here to assist you with activating your debit card. Activating your card is a simple process. Here\'s what you need to do: 1. Visit our website at Company Website URL. 2. Log in to your account using your credentials. 3. Navigate to the "Account" or "Profile" section. 4. Look for the option to activate your card. 5. Follow the prompts and provide any required information. 6. Once you\'ve'

In [19]:
import gradio as gr

In [20]:
interface = gr.Interface(
    fn=chat,          # The function to wrap with a UI
    inputs=gr.Textbox(label="Your banking question"), # Input component: a textbox for the user's prompt
    outputs=gr.Textbox(label="Chatbot response"), # Output component: a textbox for the chatbot's response
    title="Retail Banking Chatbot" # Title for the Gradio interface
)

In [21]:
import gradio as gr

interface = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Your banking question"),
    outputs=gr.Textbox(label="Chatbot response"),
    title="Retail Banking Chatbot"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://159aa808d35c8e5fac.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
